# 📝 Converter Transcrição TXT → SRT

Lê um arquivo `.txt` de transcrição (sem timecodes) e gera um `.srt` com timing estimado.

**Estratégia de timing:**  
- Distribui o tempo proporcionalmente ao número de palavras de cada segmento  
- Usa velocidade de narração configurável (padrão: 130 palavras/minuto)  
- Se você informar a duração total do áudio, o timing fica exato

## 1. Configuração

In [6]:
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES — edite aqui
# ============================================================

# Arquivo .txt de entrada
ARQUIVO_TXT = r'D:\Projetos\prompt2promolab\outputs\legendas\vide2_transcricao_escrito.txt'

# Duração total do áudio/vídeo em segundos
# None → estima automaticamente pela velocidade de narração
DURACAO_TOTAL_S = None

# Velocidade de narração (palavras por minuto)
# 120-140 → narração pausada  |  150-170 → narração normal  |  180+ → rápida
PALAVRAS_POR_MINUTO = 130

# Máximo de caracteres por linha de legenda (padrão Netflix: 42)
MAX_CHARS_POR_LINHA = 42

# Máximo de linhas por bloco SRT
MAX_LINHAS_BLOCO = 2

# Pasta de saída (None = mesma pasta do txt)
PASTA_SAIDA = None

# ============================================================

arquivo_txt = Path(ARQUIVO_TXT)
pasta_saida = Path(PASTA_SAIDA) if PASTA_SAIDA else arquivo_txt.parent
pasta_saida.mkdir(parents=True, exist_ok=True)

print(f'Entrada : {arquivo_txt}')
print(f'Saída   : {pasta_saida}')
print(f'Velocidade: {PALAVRAS_POR_MINUTO} wpm')
if DURACAO_TOTAL_S:
    print(f'Duração total: {DURACAO_TOTAL_S}s (timing exato)')
else:
    print('Duração total: estimada automaticamente')

if not arquivo_txt.exists():
    print(f'\n⚠️  Arquivo não encontrado: {arquivo_txt}')

Entrada : D:\Projetos\prompt2promolab\outputs\legendas\vide2_transcricao_escrito.txt
Saída   : D:\Projetos\prompt2promolab\outputs\legendas
Velocidade: 130 wpm
Duração total: estimada automaticamente


## 2. Ler e parsear o TXT

In [7]:
import re

raw = arquivo_txt.read_text(encoding='utf-8')

# Remove numeração de linha se existir (ex: "1\tTexto..." ou "1. Texto...")
linhas = []
for linha in raw.splitlines():
    linha = linha.strip()
    if not linha:
        continue
    # Remove prefixo numérico: "1\t", "1. ", "1) "
    linha = re.sub(r'^\d+[\t\.\)\s]+', '', linha).strip()
    if linha:
        linhas.append(linha)

texto_completo = ' '.join(linhas)
total_palavras = len(texto_completo.split())

print(f'Linhas lidas    : {len(linhas)}')
print(f'Total de palavras: {total_palavras}')
print()
print('Texto completo:')
print('-' * 60)
print(texto_completo)
print('-' * 60)

Linhas lidas    : 1
Total de palavras: 23

Texto completo:
------------------------------------------------------------
Venho apresentar a excelente casa no condomínio Monte Carlo, casa com 3 quartos, sendo uma suite master e duas semi suíte . Confira!
------------------------------------------------------------


## 3. Quebrar em segmentos de legenda

In [8]:
def quebrar_em_segmentos(texto: str, max_chars: int, max_linhas: int) -> list[str]:
    """Divide o texto em blocos respeitando max_chars por linha e max_linhas por bloco."""
    palavras = texto.split()
    segmentos = []
    linha_atual = ''
    linhas_bloco = []

    for palavra in palavras:
        candidato = f'{linha_atual} {palavra}'.strip()
        if len(candidato) <= max_chars:
            linha_atual = candidato
        else:
            if linha_atual:
                linhas_bloco.append(linha_atual)
            if len(linhas_bloco) >= max_linhas:
                segmentos.append('\n'.join(linhas_bloco))
                linhas_bloco = []
            linha_atual = palavra

    if linha_atual:
        linhas_bloco.append(linha_atual)
    if linhas_bloco:
        segmentos.append('\n'.join(linhas_bloco))

    return segmentos


segmentos = quebrar_em_segmentos(texto_completo, MAX_CHARS_POR_LINHA, MAX_LINHAS_BLOCO)

print(f'{len(segmentos)} segmentos gerados:\n')
for i, seg in enumerate(segmentos, 1):
    print(f'[{i}] {repr(seg)}')

2 segmentos gerados:

[1] 'Venho apresentar a excelente casa no\ncondomínio Monte Carlo, casa com 3'
[2] 'quartos, sendo uma suite master e duas\nsemi suíte . Confira!'


## 4. Calcular timing

In [9]:
segundos_por_palavra = 60 / PALAVRAS_POR_MINUTO

# Calcula duração proporcional de cada segmento
palavras_por_seg = [len(s.split()) for s in segmentos]
duracao_estimada = sum(palavras_por_seg) * segundos_por_palavra

duracao_total = DURACAO_TOTAL_S if DURACAO_TOTAL_S else duracao_estimada
fator_escala = duracao_total / duracao_estimada if duracao_estimada > 0 else 1.0

# Gera timestamps
timestamps = []  # lista de (inicio, fim) em segundos
cursor = 0.0
for n_palavras in palavras_por_seg:
    duracao_seg = n_palavras * segundos_por_palavra * fator_escala
    timestamps.append((cursor, cursor + duracao_seg))
    cursor += duracao_seg

print(f'Duração estimada : {duracao_estimada:.1f}s')
print(f'Duração usada    : {duracao_total:.1f}s')
if DURACAO_TOTAL_S:
    print(f'Fator de escala  : {fator_escala:.3f}')
print()
for i, ((ini, fim), seg) in enumerate(zip(timestamps, segmentos), 1):
    print(f'[{i}] {ini:.2f}s → {fim:.2f}s  |  {seg[:60]!r}')

Duração estimada : 10.6s
Duração usada    : 10.6s

[1] 0.00s → 5.54s  |  'Venho apresentar a excelente casa no\ncondomínio Monte Carlo,'
[2] 5.54s → 10.62s  |  'quartos, sendo uma suite master e duas\nsemi suíte . Confira!'


## 5. Salvar .srt

In [10]:
def para_srt_time(segundos: float) -> str:
    h  = int(segundos // 3600)
    m  = int((segundos % 3600) // 60)
    s  = int(segundos % 60)
    ms = int((segundos % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


blocos = []
for i, ((ini, fim), seg) in enumerate(zip(timestamps, segmentos), 1):
    blocos.append(f'{i}\n{para_srt_time(ini)} --> {para_srt_time(fim)}\n{seg}')

conteudo_srt = '\n\n'.join(blocos) + '\n'

arquivo_srt = pasta_saida / f'{arquivo_txt.stem.replace("_transcricao", "")}.srt'
arquivo_srt.write_text(conteudo_srt, encoding='utf-8')

print(f'✅ SRT salvo: {arquivo_srt}')
print()
print('--- Conteúdo do SRT ---')
print(conteudo_srt)

✅ SRT salvo: D:\Projetos\prompt2promolab\outputs\legendas\vide2_escrito.srt

--- Conteúdo do SRT ---
1
00:00:00,000 --> 00:00:05,538
Venho apresentar a excelente casa no
condomínio Monte Carlo, casa com 3

2
00:00:05,538 --> 00:00:10,615
quartos, sendo uma suite master e duas
semi suíte . Confira!

